# Chapter 9: Eigenvalues and eigenvectors

Companion notebook for *Linear Algebra for Data Science and Machine Learning: Worked Examples and Python* by Dr. Sajjad Shaukat Jamal (Skills Outlet).

The cells below contain every Python example from Chapter 9, in the order they appear in the book. Work each example by hand first, then run the cell to check your result. Cells within a chapter build on each other, so run them from top to bottom.

## 9.11  Python checks for eigenpairs and PCA

For a real symmetric matrix, use the symmetric eigensolver eigh. It returns eigenvalues in ascending order, with the corresponding eigenvectors in columns. For PCA, reverse their order together so the largest variance appears first.

In [1]:
import numpy as np
A = np.array([[2., 1.], [1., 2.]])
values, vectors = np.linalg.eigh(A)
print(values)  # [1. 3.]
assert np.allclose(A @ vectors, vectors * values)
assert np.allclose(vectors.T @ vectors, np.eye(2))
assert np.allclose(
    vectors @ np.diag(values) @ vectors.T, A
)

[1. 3.]


Multiplying vectors by the one-dimensional values array scales its columns. The first check verifies Av = λv for both columns. The next checks orthonormality, and the last reconstructs A. Use tolerances for floating-point comparisons.

In [2]:
X = np.array([[2., 2.], [-2., -2.],
              [1., -1.], [-1., 1.]])
mean = X.mean(axis=0)
Xc = X - mean
S = Xc.T @ Xc / (len(Xc) - 1)
values, Q = np.linalg.eigh(S)
order = np.argsort(values)[::-1]
values, Q = values[order], Q[:, order]
ratios = values / values.sum()
scores = Xc @ Q[:, :1]
reconstructed = scores @ Q[:, :1].T + mean
print(values, ratios)
assert np.allclose(ratios, [0.8, 0.2])
assert np.isclose(np.sum((X-reconstructed)**2), 4)

[5.33333333 1.33333333] [0.8 0.2]


Expected eigenvalues are approximately (5.333333, 1.333333). The ratio formula assumes positive total variance; if every centered observation is zero, there is no variance to divide into proportions. In a predictive workflow, estimate means and PCA directions from training data only, then apply them to held-out observations.

## 9.15  Optional extension: Covariance and loss geometry

### Example 21  A covariance-weighted distance

For covariance Σ = diag(4, 1) and displacement d = (2, 1), the squared Mahalanobis distance is dᵀΣ⁻¹d = 2²/4 + 1²/1 = 2. One unit along a high-variance direction contributes less. In a nonsingular multivariate Gaussian model, the density’s exponent is −½dᵀΣ⁻¹d. This connects covariance geometry to Gaussian models; it does not prove that arbitrary data are Gaussian.

In [3]:
C = np.array([[1., 1.], [1., 1.]])
assert np.allclose(np.linalg.eigvalsh(C), [0., 2.])
d = np.array([2., 1.])
Sigma = np.diag([4., 1.])
distance2 = d @ np.linalg.solve(Sigma, d)
assert np.isclose(distance2, 2.)